In [2]:
import numpy as np
import matplotlib.pyplot as plt
from forTP1 import Bandit

In [3]:
K=10
T=500

## <font color='red'>Greedy and Epsilon-Greedy</font>

In [71]:
class myAlgo_greedy:
    def __init__(self,bandi,T):
        self.mybandi=bandi
        self.T=T
        self.N=(self.T/self.mybandi.K)**(2/3)*2
        self.count=np.zeros(self.mybandi.K)
        self.rewards=[]
        self.bandis=[]

    def greddy_and_epsi_greedy(self,epsilon=None):
        qt=np.zeros(self.mybandi.K)
        # Exploration phase
        for b in range(self.mybandi.K):
            for n in range(1,int(self.N+1)):
                reward = self.mybandi.get_arm(b)
                self.rewards.append(reward)
                qt[b] += (reward - qt[b]) / n
                self.bandis.append(b)
       # choose best hand
        if epsilon is None or np.random.rand() > epsilon:
          
          best_bandi = np.argmax(qt)
        else:
            best_bandi= np.random.randint(self.mybandi.K)

        #Exploitation phase
        for _ in range(int(self.T-self.N)):
            reward = self.mybandi.get_arm(best_bandi)
            self.rewards.append(reward)
            self.bandis.append(best_bandi)

        return self.rewards, self.bandis


In [72]:
bandi=Bandit(K)
algo=myAlgo_greedy(bandi,T)
print(f'Episol_Greedy: {algo.greddy_and_epsi_greedy(0.1)}')
print(f'Greedy: {algo.greddy_and_epsi_greedy()}')

Episol_Greedy: ([np.int64(9), np.int64(6), np.int64(8), np.int64(7), np.int64(10), np.int64(9), np.int64(6), np.int64(9), np.int64(6), np.int64(6), np.int64(7), np.int64(5), np.int64(9), np.int64(4), np.int64(14), np.int64(9), np.int64(6), np.int64(6), np.int64(9), np.int64(4), np.int64(10), np.int64(5), np.int64(6), np.int64(9), np.int64(4), np.int64(3), np.int64(3), np.int64(10), np.int64(5), np.int64(9), np.int64(5), np.int64(12), np.int64(6), np.int64(10), np.int64(9), np.int64(11), np.int64(7), np.int64(6), np.int64(9), np.int64(5), np.int64(14), np.int64(13), np.int64(5), np.int64(8), np.int64(7), np.int64(8), np.int64(11), np.int64(5), np.int64(4), np.int64(9), np.int64(6), np.int64(5), np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(11), np.int64(9), np.int64(10), np.int64(11), np.int64(10), np.int64(9), np.int64(14), np.int64(8), np.int64(11), np.int64(16), np.int64(9), np.int64(8), np.int64(12), np.int64(17), np.int64(11), np.int64(6), np.int64(8), np.int64(12),

## <font color='red'>Sucessive Elimination</font>

In [40]:
class SuccessiveElimination:
    def __init__(self, bandi, delta=0.1):
        self.bandi = bandi
        self.counts = np.zeros(self.bandi.K)    
        self.values = np.zeros(self.bandi.K)    
        self.actice_bandi = list(range(self.bandi.K))
        self.delta = delta
        self.rewards = []
        self.T=T
        

    def ucb(self, bandi):
        if self.counts[bandi] == 0:
            return float('inf')
        rt = 2 * np.log(self.T) / self.counts[bandi]
        return self.values[bandi] + rt

    def lcb(self, bandi):
        if self.counts[bandi] == 0:
            return -float('inf')
        rt = 2 * np.log(self.T) / self.counts[bandi]
        return self.values[bandi] - rt


    def run(self):
        t = 0
        while t < self.T and len(self.actice_bandi) > 1:
            
            for band in self.actice_bandi.copy():
                reward = self.bandi.get_arm(band)
                self.counts[band] += 1
                n = self.counts[band]
                self.values[band] += (reward - self.values[band]) / n
                self.rewards.append(reward)
                t += 1
                if t >= self.T:
                    break

            
            for band in self.actice_bandi.copy():
                for other_bandi in self.actice_bandi:
                    if band != other_bandi and self.ucb(band) < self.lcb(other_bandi):
                        if band in self.actice_bandi:
                            self.actice_bandi.remove(band)
                            break

        #Exploitation
        if self.actice_bandi:
            best_bandi = self.actice_bandi[0]
            while t < self.T:
                reward = self.bandi.get_arm(best_bandi)
                self.counts[best_bandi] += 1
                n = self.counts[best_bandi]
                self.values[best_bandi] += (reward - self.values[best_bandi]) / n
                self.rewards.append(reward)
                t += 1

        return self.rewards, self.actice_bandi

In [41]:
suc=SuccessiveElimination(bandi,0.05)
print(suc.run())

([np.int64(13), np.int64(11), np.int64(8), np.int64(4), np.int64(8), np.int64(5), np.int64(11), np.int64(9), np.int64(10), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(7), np.int64(9), np.int64(10), np.int64(18), np.int64(17), np.int64(13), np.int64(10), np.int64(11), np.int64(7), np.int64(9), np.int64(6), np.int64(10), np.int64(8), np.int64(15), np.int64(12), np.int64(8), np.int64(6), np.int64(13), np.int64(8), np.int64(11), np.int64(7), np.int64(5), np.int64(15), np.int64(11), np.int64(10), np.int64(7), np.int64(14), np.int64(6), np.int64(14), np.int64(15), np.int64(11), np.int64(9), np.int64(10), np.int64(6), np.int64(13), np.int64(6), np.int64(6), np.int64(9), np.int64(14), np.int64(9), np.int64(13), np.int64(14), np.int64(14), np.int64(20), np.int64(18), np.int64(13), np.int64(16), np.int64(9), np.int64(12), np.int64(15), np.int64(13), np.int64(16), np.int64(14), np.int64(15), np.int64(10), np.int64(19), np.int64(10), np.int64(13), np.int64(13), np.int64(18), np

## <font color='red'>UCB1</font>

In [ ]:
class UCB1:
    def __init__(self, bandit,T):
        self.bandit = bandit
        self.T=T        
        self.K = bandit.K
        self.counts = np.zeros(self.K)    
        self.values = np.zeros(self.K)   
        self.total_rewards = []
        self.actions = []

    def run(self):
      
        for band in range(self.K):
            reward = self.bandit.get_arm(band)
            self.counts[band] += 1
            self.values[band] += reward
            self.total_rewards.append(reward)
            self.actions.append(band)

        for t in range(self.K, self.T):
            
            ucb_values = np.zeros(self.K)
            for band in range(self.K):
                rt = 2 * np.log(self.T) / self.counts[band]
                ucb_values[band] = self.values[band]/self.counts[band] + rt

            action = np.argmax(ucb_values)
            reward = self.bandit.get_arm(action)

            self.counts[action] += 1
            self.values[action] += reward
            self.total_rewards.append(reward)
            self.actions.append(action)

        return self.total_rewards,self.actions


In [39]:
uc=UCB1(bandi,T)
print(uc.run())

[np.int64(12), np.int64(7), np.int64(5), np.int64(7), np.int64(10), np.int64(3), np.int64(14), np.int64(6), np.int64(9), np.int64(8), np.int64(16), np.int64(15), np.int64(9), np.int64(7), np.int64(14), np.int64(4), np.int64(10), np.int64(6), np.int64(13), np.int64(13), np.int64(3), np.int64(10), np.int64(15), np.int64(13), np.int64(13), np.int64(16), np.int64(8), np.int64(13), np.int64(12), np.int64(6), np.int64(11), np.int64(4), np.int64(12), np.int64(16), np.int64(14), np.int64(21), np.int64(12), np.int64(11), np.int64(12), np.int64(11), np.int64(11), np.int64(15), np.int64(11), np.int64(9), np.int64(9), np.int64(17), np.int64(23), np.int64(15), np.int64(15), np.int64(17), np.int64(16), np.int64(15), np.int64(16), np.int64(13), np.int64(14), np.int64(13), np.int64(12), np.int64(19), np.int64(13), np.int64(13), np.int64(12), np.int64(18), np.int64(17), np.int64(13), np.int64(16), np.int64(14), np.int64(17), np.int64(14), np.int64(15), np.int64(11), np.int64(12), np.int64(19), np.int64

## <font color='yellow'>Nombre de round </font> : T = 500  pour visualisation claire du regret
## <font color='yellow'>fontε-greedy </font>: ε = 0.1 

## <font color='yellow'>delta(Successive Elimination) </font>= 0.05 c'est à dire 95% de confiance pour les intervalles


In [89]:

def estimate_regret_greedy(algo,T=500, n_runs=1000, eps=0.1):
    # estimer les moyennes réelles
    true_means = [np.mean([algo.mybandi.get_arm(a) for _ in range(n_runs)]) for a in range(algo.mybandi.K)]
    best_rewar = max(true_means)
    cumulative_regrets = np.zeros(T)

    
    algo=myAlgo_greedy(Bandit(algo.mybandi.K),T)
    rewards, bandi = algo.greddy_and_epsi_greedy(eps)  
    
    regrets=best_rewar-np.array([true_means[a] for a in bandi])
    cumulative_regrets = np.zeros(len(bandi))
    cumulative_regrets= np.cumsum(regrets)
    for _ in range(n_runs):
            algo=myAlgo_greedy(Bandit(algo.mybandi.K),T)
            rewards, bandi = algo.greddy_and_epsi_greedy(eps)  
    
            regrets=best_rewar-np.array([true_means[a] for a in bandi])
            
            cumulative_regrets+= np.cumsum(regrets)
    avg_regret = cumulative_regrets / n_runs
    return avg_regret


In [90]:
bandi=Bandit(K)
greedy=myAlgo_greedy(bandi,T)
estimate_regret_greedy(greedy,500, 1000,0.1)

array([   6.004999,   12.009998,   18.014997,   24.019996,   30.024995,
         36.029994,   42.034993,   48.039992,   54.044991,   60.04999 ,
         66.054989,   72.059988,   78.064987,   84.069986,   90.074985,
         96.079984,  102.084983,  108.089982,  114.094981,  120.09998 ,
        126.104979,  132.109978,  138.114977,  144.119976,  150.124975,
        156.129974,  162.134973,  164.017854,  165.900735,  167.783616,
        169.666497,  171.549378,  173.432259,  175.31514 ,  177.198021,
        179.080902,  180.963783,  182.846664,  184.729545,  186.612426,
        188.495307,  190.378188,  192.261069,  194.14395 ,  196.026831,
        197.909712,  199.792593,  201.675474,  203.558355,  205.441236,
        207.324117,  209.206998,  211.089879,  212.97276 ,  217.773556,
        222.574352,  227.375148,  232.175944,  236.97674 ,  241.777536,
        246.578332,  251.379128,  256.179924,  260.98072 ,  265.781516,
        270.582312,  275.383108,  280.183904,  284.9847  ,  289.